In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

WHEEL_DIRS = [
    "/kaggle/input/datasets/mayukh18/nemotron-packages/packages",
    "/kaggle/input/datasets/llkh0a/rtx-wheels/wheels",
]

TARGET_SITE = Path("/kaggle/working/eval_site")

WHEEL_REQS = [
    "vllm",
    "openai_harmony",
    "openai",
    "transformers==4.57.6",
    "pandas",
    "numpy",
    "sympy",
    "mpmath",
    "safetensors",
    "tqdm",
    "peft",
    "accelerate",
]

def install_offline_to_target():
    if TARGET_SITE.exists():
        shutil.rmtree(TARGET_SITE)
    TARGET_SITE.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
    ]

    for d in WHEEL_DIRS:
        assert Path(d).is_dir(), f"Missing wheel dir: {d}"
        cmd += ["--find-links", d]

    cmd += [
        "--only-binary=:all:",
        "--target", str(TARGET_SITE),
        "--upgrade",
        *WHEEL_REQS,
    ]

    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)


install_offline_to_target()

sys.path.insert(0, str(TARGET_SITE))

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

_old_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = str(TARGET_SITE) + (":" + _old_pythonpath if _old_pythonpath else "")

print("ОК: офлайн-пакеты установлены в", TARGET_SITE)

In [ ]:
from pathlib import Path
import os
import json

MODEL_PATH_HINT = "/kaggle/input/datasets/tkaggl/merged-dapo40/dapo40-merged"

BUNDLE_ROOT_HINTS = []

TIKTOKEN_ENCODINGS_HINTS = [
    "/kaggle/input/datasets/tkaggl/dapo40/tiktoken_encodings"
]

EVAL_CSV_PATHS = [
    "/kaggle/input/datasets/tkaggl/imhobench/imhobench.csv"
]

OUTPUT_DIR = Path("/kaggle/working/gptoss20_vllm_eval2_dapo-dapo40")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_BASE = True
RUN_LORA = False

BASE_SERVED_MODEL_NAME = "gpt-oss-20b-dapo40"
LORA_MODEL_NAME = "olymp"

LORA_RANK = 32

SERVE_PORT = 8000
BASE_SEED = 42

N_PROBLEMS = None
SAMPLES_PER_PROBLEM = 16

WORKERS = 8
MAX_NUM_SEQS = 32
STREAM_INTERVAL = 20
GPU_MEMORY_UTILIZATION = 0.92

TEMPERATURE = 1.0
TOP_P = 1.0
MIN_P = None

MAX_TURNS = 96
CONTEXT_WINDOW = 32_768
CONTEXT_RESERVE = 128
GENERATION_CAP = 32_768

PYTHON_TIMEOUT = 8.0
ATTEMPT_TIMEOUT_SECONDS = 900

STOP_MODE = "strings"
STOP_PREF_ORDER = ["strings", "strings_include", "token_ids"]
STOP_PREFLIGHT_MAX_TOKENS = 512
ASSERT_STRINGS_STOP_PREFLIGHT = True

REQUIRE_MXFP4 = True

SYSTEM_IDENTITY = "You are GPT-OSS, a careful mathematical reasoning model."

DEVELOPER_PROMPT = """
You are solving hard olympiad-style math problems.

Rules:
1. Work carefully and avoid arithmetic mistakes.
2. Explore, plan, execute, and verify.
3. The final answer must be a non-negative integer from 0 to 99999.
4. Put the final answer inside \\boxed{...}.
5. If you are unsure, still produce your best final boxed answer.

Be concise when possible. Do not ramble needlessly.
""".strip()

TOOL_PROMPT_BASELINE = """
Use the python tool only when it meaningfully helps:
- exact arithmetic
- modular arithmetic
- brute force on small search spaces
- symbolic checks with sympy
- quick verification

Keep code short and use print().
Code should support reasoning, not replace it.
""".strip()

In [ ]:
from pathlib import Path
import os
import json


def resolve_model_path(model_path_hint: str) -> str:
    p = Path(model_path_hint)
    if p.is_dir() and (p / "config.json").exists() and (p / "tokenizer.json").exists():
        return str(p)

    if p.is_dir():
        for root, dirs, files in os.walk(p):
            files = set(files)
            if "config.json" in files and "tokenizer.json" in files:
                if "model.safetensors.index.json" in files or any(f.endswith(".safetensors") for f in files):
                    return root

    raise FileNotFoundError(f"Could not resolve model path from {model_path_hint}")


def resolve_bundle_root(hints: list[str]) -> Path | None:
    for h in hints:
        p = Path(h)
        if not p.exists():
            continue

        peft = p / "peft_olymp_adapter"
        tik = p / "tiktoken_encodings"

        if (
            (peft / "adapter_config.json").exists()
            and (peft / "adapter_model.safetensors").exists()
            and (tik / "o200k_base.tiktoken").exists()
            and (tik / "cl100k_base.tiktoken").exists()
        ):
            return p

    return None


def resolve_tiktoken_encodings(hints: list[str]) -> Path | None:
    for h in hints:
        p = Path(h)
        if (
            p.is_dir()
            and (p / "o200k_base.tiktoken").exists()
            and (p / "cl100k_base.tiktoken").exists()
        ):
            return p
    return None


MODEL_PATH = Path(resolve_model_path(MODEL_PATH_HINT))

cfg = json.loads((MODEL_PATH / "config.json").read_text())
print("тип модели:", cfg.get("model_type"))
print("архитектуры:", cfg.get("architectures"))
print("конфигурация квантования:", cfg.get("quantization_config"))

if REQUIRE_MXFP4:
    qcfg = cfg.get("quantization_config") or {}
    assert qcfg.get("quant_method") == "mxfp4", f"Expected MXFP4 base model, got: {qcfg}"

BUNDLE_ROOT = resolve_bundle_root(BUNDLE_ROOT_HINTS)

if RUN_LORA:
    assert BUNDLE_ROOT is not None, "RUN_LORA=True, but no valid offline_lora_bundle found"

if BUNDLE_ROOT is not None:
    PEFT_ADAPTER_PATH = BUNDLE_ROOT / "peft_olymp_adapter"
    TIKTOKEN_ENCODINGS_PATH = BUNDLE_ROOT / "tiktoken_encodings"

    assert (PEFT_ADAPTER_PATH / "adapter_config.json").exists()
    assert (PEFT_ADAPTER_PATH / "adapter_model.safetensors").exists()
    assert (TIKTOKEN_ENCODINGS_PATH / "o200k_base.tiktoken").exists()
    assert (TIKTOKEN_ENCODINGS_PATH / "cl100k_base.tiktoken").exists()

    os.environ["TIKTOKEN_ENCODINGS_BASE"] = str(TIKTOKEN_ENCODINGS_PATH)
    os.environ.pop("TIKTOKEN_RS_CACHE_DIR", None)

    print("корень бандла:", BUNDLE_ROOT)
    print("путь PEFT-адаптера:", PEFT_ADAPTER_PATH)
    print("путь кодировок tiktoken:", TIKTOKEN_ENCODINGS_PATH)
else:
    PEFT_ADAPTER_PATH = None
    TIKTOKEN_ENCODINGS_PATH = resolve_tiktoken_encodings(TIKTOKEN_ENCODINGS_HINTS)
    assert TIKTOKEN_ENCODINGS_PATH is not None, (
        "No bundle found and no tiktoken_encodings found. "
        "Provide offline_lora_bundle or set TIKTOKEN_ENCODINGS_HINTS."
    )
    os.environ["TIKTOKEN_ENCODINGS_BASE"] = str(TIKTOKEN_ENCODINGS_PATH)
    os.environ.pop("TIKTOKEN_RS_CACHE_DIR", None)
    print("бандл LoRA не найден. запуск только базы.")
    print("путь кодировок tiktoken:", TIKTOKEN_ENCODINGS_PATH)

if not RUN_LORA:
    PEFT_ADAPTER_PATH = None

In [ ]:
import os
import re
import ast
import sys
import json
import time
import queue
import uuid
import base64
import signal
import hashlib
import shutil
import socket
import contextlib
import subprocess
import threading
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI

from openai_harmony import (
    Author,
    Conversation,
    HarmonyEncodingName,
    Message,
    Role,
    SystemContent,
    ReasoningEffort,
    ToolNamespaceConfig,
    TextContent,
    load_harmony_encoding,
)


def stable_seed(*parts, base_seed=BASE_SEED):
    s = "|".join(str(x) for x in parts)
    h = hashlib.sha256(s.encode("utf-8")).hexdigest()
    return (int(h[:12], 16) + int(base_seed)) % (2**31 - 1)


def extract_answer_strict(text: str):
    if not text:
        return None
    matches = re.findall(r"\\boxed\{\s*([0-9,]+)\s*\}", text)
    if not matches:
        return None
    try:
        v = int(matches[-1].replace(",", ""))
    except Exception:
        return None
    return v if 0 <= v <= 99999 else None


def extract_answer_loose(text: str):
    if not text:
        return None

    patterns = [
        r"\\boxed\s*\{\s*([0-9,]+)\s*\}",
        r"boxed\s*\(\s*([0-9,]+)\s*\)",
        r"final\s+answer\s*(?:is|=|:)\s*([0-9,]+)",
        r"answer\s*(?:is|=|:)\s*([0-9,]+)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            try:
                v = int(matches[-1].replace(",", ""))
                if 0 <= v <= 99999:
                    return v
            except Exception:
                pass

    tail = text[-1500:]
    matches = re.findall(r"(?<!\d)([0-9]{1,5})(?!\d)", tail)
    if matches:
        try:
            v = int(matches[-1])
            if 0 <= v <= 99999:
                return v
        except Exception:
            pass

    return None


def majority_answer(xs):
    clean = []
    for x in xs:
        if x is None:
            continue
        try:
            if pd.isna(x):
                continue
        except Exception:
            pass
        try:
            clean.append(int(x))
        except Exception:
            pass

    if not clean:
        return None, 0

    return Counter(clean).most_common(1)[0]


def harmony_message_text(msg) -> str:
    c = getattr(msg, "content", None)

    if c is None:
        return ""

    if isinstance(c, str):
        return c

    if hasattr(c, "text"):
        return c.text or ""

    if isinstance(c, list):
        parts = []
        for x in c:
            if isinstance(x, str):
                parts.append(x)
            elif hasattr(x, "text"):
                parts.append(x.text or "")
            elif hasattr(x, "content"):
                parts.append(str(x.content))
            else:
                parts.append(str(x))
        return "".join(parts)

    return str(c)


def make_conversation(messages):
    try:
        return Conversation.from_messages(messages)
    except Exception:
        return Conversation(messages=messages)


def load_eval_df():
    dfs = []
    for p in EVAL_CSV_PATHS:
        path = Path(p)
        if not path.exists():
            print("пропущен отсутствующий файл:", p)
            continue
        df = pd.read_csv(path)
        df["_source_csv"] = str(path)
        dfs.append(df)

    assert dfs, f"No eval CSV found in: {EVAL_CSV_PATHS}"
    df = pd.concat(dfs, ignore_index=True)

    problem_col_candidates = [
        "problem", "question", "prompt", "statement", "Problem", "Question", "text"
    ]
    answer_col_candidates = [
        "answer", "gold_answer", "Answer", "target", "label"
    ]
    id_col_candidates = [
        "problem_id", "id", "ID", "uid"
    ]

    problem_col = next((c for c in problem_col_candidates if c in df.columns), None)
    answer_col = next((c for c in answer_col_candidates if c in df.columns), None)
    id_col = next((c for c in id_col_candidates if c in df.columns), None)

    assert problem_col is not None, f"Cannot find problem column. Columns: {list(df.columns)}"
    assert answer_col is not None, f"Cannot find answer column. Columns: {list(df.columns)}"

    out = pd.DataFrame()
    out["problem"] = df[problem_col].astype(str)
    out["answer"] = df[answer_col].astype(int)

    if id_col is not None:
        out["problem_id"] = df[id_col].astype(str)
    else:
        out["problem_id"] = [f"problem_{i:05d}" for i in range(len(df))]

    out["_source_csv"] = df["_source_csv"].astype(str)

    if N_PROBLEMS is not None:
        out = out.head(int(N_PROBLEMS)).copy()

    out = out.reset_index(drop=True)

    print("загружено строк:", len(out))
    display(out.head())
    return out


eval_df = load_eval_df()

In [ ]:
ERROR_MARKERS = (
    "[ERROR]",
    "Traceback",
    "SyntaxError",
    "IndentationError",
    "NameError",
    "TypeError",
    "ValueError",
    "ImportError",
    "ModuleNotFoundError",
    "ZeroDivisionError",
    "RecursionError",
    "OverflowError",
    "AssertionError",
)

SENTINEL_RE = re.compile(r"<\|(?:call|end|return|message|channel)\|>")


def clean_tool_code(code: str) -> str:
    cleaned = (code or "").strip()
    cleaned = re.split(r"<\|(?:call|end|return)\|>", cleaned)[0].strip()
    cleaned = re.sub(r"^```(?:python)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return cleaned.strip()


def rewrite_last_expr_to_print(code: str) -> tuple[str, bool]:
    try:
        tree = ast.parse(code, mode="exec")
    except SyntaxError:
        return code, False

    if not tree.body:
        return code, False

    last = tree.body[-1]
    if not isinstance(last, ast.Expr):
        return code, False

    tmp_name = "__tool_last_expr__"

    assign = ast.Assign(
        targets=[ast.Name(id=tmp_name, ctx=ast.Store())],
        value=last.value,
    )

    print_if_not_none = ast.If(
        test=ast.Compare(
            left=ast.Name(id=tmp_name, ctx=ast.Load()),
            ops=[ast.IsNot()],
            comparators=[ast.Constant(value=None)],
        ),
        body=[
            ast.Expr(
                value=ast.Call(
                    func=ast.Name(id="print", ctx=ast.Load()),
                    args=[ast.Name(id=tmp_name, ctx=ast.Load())],
                    keywords=[],
                )
            )
        ],
        orelse=[],
    )

    tree.body[-1:] = [assign, print_if_not_none]
    ast.fix_missing_locations(tree)

    try:
        return ast.unparse(tree), True
    except Exception:
        return code, False


def is_tool_error(output: str) -> bool:
    output = output or ""
    return any(marker in output for marker in ERROR_MARKERS)


class SubprocessPythonSandbox:
    def __init__(self, timeout: float = PYTHON_TIMEOUT):
        self.timeout = float(timeout)
        self.proc = None
        self.q = None
        self._reader_thread = None
        self._start()

    def _server_code(self) -> str:
        return r'''
import sys, json, base64, io, traceback, contextlib, math, itertools, collections

try:
    import numpy as np
except Exception:
    pass

try:
    import sympy as sp
except Exception:
    pass

try:
    import mpmath
    mpmath.mp.dps = 64
except Exception:
    pass

try:
    import pulp
    _OrigPULP_CBC_CMD = pulp.PULP_CBC_CMD

    class QuietPULP_CBC_CMD(_OrigPULP_CBC_CMD):
        def __init__(self, *args, **kwargs):
            kwargs.setdefault("msg", False)
            super().__init__(*args, **kwargs)

    pulp.PULP_CBC_CMD = QuietPULP_CBC_CMD

    try:
        pulp.LpSolverDefault.msg = False
    except Exception:
        pass
except Exception:
    pass

_GLOBALS = globals()

print("__SANDBOX_READY__", flush=True)

for _line in sys.stdin:
    _line = _line.strip()
    if not _line:
        continue

    try:
        _req = json.loads(_line)
        _code = base64.b64decode(_req["code_b64"]).decode("utf-8")
        _stdout = io.StringIO()
        _stderr = io.StringIO()

        with contextlib.redirect_stdout(_stdout), contextlib.redirect_stderr(_stderr):
            try:
                exec(compile(_code, "<python_tool>", "exec"), _GLOBALS, _GLOBALS)
            except BaseException:
                traceback.print_exc(file=_stderr)

        _out = _stdout.getvalue()
        _err = _stderr.getvalue()
        _text = (_out.rstrip() + ("\n" if _out.rstrip() and _err.rstrip() else "") + _err.rstrip()).strip()

        if not _text:
            _text = "[WARN] No output"

        _resp = {
            "id": _req.get("id"),
            "output_b64": base64.b64encode(_text.encode("utf-8", errors="replace")).decode("ascii"),
        }

    except BaseException:
        _text = traceback.format_exc()
        _resp = {
            "id": None,
            "output_b64": base64.b64encode(_text.encode("utf-8", errors="replace")).decode("ascii"),
        }

    print("__SANDBOX_RESULT__" + json.dumps(_resp, ensure_ascii=False), flush=True)
'''

    def _start(self):
        self.close()

        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
        env["MPLBACKEND"] = "Agg"

        self.proc = subprocess.Popen(
            [sys.executable, "-u", "-c", self._server_code()],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
            start_new_session=True,
        )

        self.q = queue.Queue()

        def _reader():
            try:
                for line in self.proc.stdout:
                    self.q.put(line)
            except Exception as exc:
                self.q.put("__SANDBOX_RESULT__" + json.dumps({
                    "id": None,
                    "output_b64": base64.b64encode(f"[READER_ERROR] {exc}".encode()).decode("ascii"),
                }) + "\n")

        self._reader_thread = threading.Thread(target=_reader, daemon=True)
        self._reader_thread.start()

        deadline = time.time() + 30.0
        while time.time() < deadline:
            try:
                line = self.q.get(timeout=0.1)
            except queue.Empty:
                if self.proc.poll() is not None:
                    break
                continue

            if "__SANDBOX_READY__" in line:
                return

        with contextlib.suppress(Exception):
            os.killpg(os.getpgid(self.proc.pid), signal.SIGKILL)
        self.proc = None
        raise RuntimeError("SubprocessPythonSandbox failed to become ready")

    def execute(self, code: str, timeout: float | None = None) -> tuple[str, dict]:
        raw_code = code or ""
        cleaned = clean_tool_code(raw_code)

        flags = {
            "had_harmony_sentinel": bool(SENTINEL_RE.search(raw_code)),
            "had_markdown_fence": bool(re.search(r"^\s*```", raw_code.strip())),
            "rewrote_last_expr": False,
            "blocked_exit": False,
            "raw_code_head": raw_code[:500],
            "raw_code_tail": raw_code[-500:],
            "clean_code_head": cleaned[:500],
            "clean_code_tail": cleaned[-500:],
            "raw_code_len": len(raw_code),
            "clean_code_len": len(cleaned),
        }

        if self.proc is None or self.proc.poll() is not None:
            self._start()

        if not cleaned:
            output = "[ERROR] Empty python tool code"
            return output, {**flags, "is_error": True, "is_no_output": False}

        if re.search(r"^\s*(exit|quit)\s*\(\s*\)\s*$", cleaned, flags=re.MULTILINE):
            output = "[ERROR] exit()/quit() are disabled. Run normal Python code instead."
            flags["blocked_exit"] = True
            return output, {**flags, "is_error": True, "is_no_output": False}

        cleaned, rewrote = rewrite_last_expr_to_print(cleaned)
        flags["rewrote_last_expr"] = bool(rewrote)
        flags["clean_code_head"] = cleaned[:500]
        flags["clean_code_tail"] = cleaned[-500:]
        flags["clean_code_len"] = len(cleaned)

        effective_timeout = float(timeout or self.timeout)
        req_id = uuid.uuid4().hex

        req = {
            "id": req_id,
            "code_b64": base64.b64encode(cleaned.encode("utf-8")).decode("ascii"),
        }

        try:
            self.proc.stdin.write(json.dumps(req, ensure_ascii=False) + "\n")
            self.proc.stdin.flush()
        except Exception as exc:
            self._start()
            output = f"[ERROR] sandbox stdin failed: {exc}"
            return output, {**flags, "is_error": True, "is_no_output": False}

        deadline = time.time() + effective_timeout
        output = None

        while time.time() < deadline:
            try:
                line = self.q.get(timeout=0.1)
            except queue.Empty:
                if self.proc.poll() is not None:
                    break
                continue

            if not line.startswith("__SANDBOX_RESULT__"):
                continue

            try:
                payload = json.loads(line[len("__SANDBOX_RESULT__"):])
                if payload.get("id") not in {req_id, None}:
                    continue
                output = base64.b64decode(payload["output_b64"]).decode("utf-8", errors="replace")
                if not output.strip():
                    output = "[WARN] No output"
                break
            except Exception as exc:
                output = f"[ERROR] bad sandbox response: {exc}"
                break

        if output is None:
            with contextlib.suppress(Exception):
                os.killpg(os.getpgid(self.proc.pid), signal.SIGKILL)
            self.proc = None
            output = f"[ERROR] Execution timed out after {effective_timeout} seconds"

        return output, {
            **flags,
            "is_error": bool(is_tool_error(output)),
            "is_no_output": bool("[WARN] No output" in output),
        }

    def close(self):
        if getattr(self, "proc", None) is not None:
            with contextlib.suppress(Exception):
                os.killpg(os.getpgid(self.proc.pid), signal.SIGTERM)
            with contextlib.suppress(Exception):
                self.proc.wait(timeout=2)
            with contextlib.suppress(Exception):
                os.killpg(os.getpgid(self.proc.pid), signal.SIGKILL)
        self.proc = None

    def __del__(self):
        self.close()

In [ ]:
@dataclass
class ExperimentConfig:
    label: str
    served_model_name: str
    use_tools: bool = True
    reasoning_effort: str = "medium"
    context_window: int = CONTEXT_WINDOW
    generation_cap: int = GENERATION_CAP
    max_turns: int = MAX_TURNS
    tool_prompt: str = TOOL_PROMPT_BASELINE
    samples: int = SAMPLES_PER_PROBLEM


EXPERIMENTS = []

if RUN_BASE:
    EXPERIMENTS.append(
        ExperimentConfig(
            label="imho_sb40_tool_medium_32k",
            served_model_name=BASE_SERVED_MODEL_NAME,
            use_tools=True,
            reasoning_effort="medium",
        )
    )

if RUN_LORA:
    EXPERIMENTS.append(
        ExperimentConfig(
            label="lora_tool_medium_32k",
            served_model_name=LORA_MODEL_NAME,
            use_tools=True,
            reasoning_effort="medium",
        )
    )

assert EXPERIMENTS, "No experiments selected"

GLOBAL_MAX_CONTEXT = max(e.context_window for e in EXPERIMENTS)


def harmony_stop_sequences(encoding) -> list[str]:
    stop_strings = []
    for token in encoding.stop_tokens_for_assistant_actions():
        try:
            s = encoding.decode_utf8([token])
        except Exception:
            continue
        if s and s not in stop_strings:
            stop_strings.append(s)
    return stop_strings


def find_free_port(preferred: int):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def reasoning_effort_from_str(s: str):
    s = (s or "medium").lower().strip()
    if s == "low":
        return ReasoningEffort.LOW
    if s == "medium":
        return ReasoningEffort.MEDIUM
    if s == "high":
        return ReasoningEffort.HIGH
    raise ValueError(f"Unknown reasoning effort: {s}")


class GPTOSSRunner:
    def __init__(self):
        self.port = find_free_port(SERVE_PORT)
        self.proc = None
        self.client = OpenAI(
            base_url=f"http://127.0.0.1:{self.port}/v1",
            api_key="EMPTY",
            timeout=SERVER_TIMEOUT if "SERVER_TIMEOUT" in globals() else 240,
        )

        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = list(self.encoding.stop_tokens_for_assistant_actions())
        self.stop_strings = harmony_stop_sequences(self.encoding)
        self.stop_mode_by_model = {}

        print("стоп-строки:", self.stop_strings)
        print("стоп-токены:", self.stop_token_ids)

        self._start_server()
        self._wait_for_server()
        self._assert_served_models()

    def _assert_served_models(self):
        models = self.client.models.list()
        ids = {m.id for m in models.data}
        print("список моделей:", sorted(ids))

        assert BASE_SERVED_MODEL_NAME in ids, (
            f"Base served model {BASE_SERVED_MODEL_NAME!r} is not listed by vLLM: {ids}"
        )
        if RUN_LORA:
            assert LORA_MODEL_NAME in ids, (
                f"LoRA served model {LORA_MODEL_NAME!r} is not listed by vLLM: {ids}"
            )

    def _start_server(self):
        cmd = [
            sys.executable,
            "-m",
            "vllm.entrypoints.openai.api_server",

            "--model", str(MODEL_PATH),
            "--served-model-name", BASE_SERVED_MODEL_NAME,

            "--host", "0.0.0.0",
            "--port", str(self.port),
            "--tensor-parallel-size", "1",
            "--dtype", "bfloat16",
            "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
            "--max-model-len", str(GLOBAL_MAX_CONTEXT),
            "--max-num-seqs", str(MAX_NUM_SEQS),
            "--stream-interval", str(STREAM_INTERVAL),
            "--disable-log-stats",
            "--seed", str(BASE_SEED),
            "--generation-config", "vllm",
        ]

        if RUN_LORA:
            assert PEFT_ADAPTER_PATH is not None
            cmd += [
                "--enable-lora",
                "--max-lora-rank", str(LORA_RANK),
                "--lora-modules", f"{LORA_MODEL_NAME}={PEFT_ADAPTER_PATH}",
            ]

        print("запуск сервера vLLM:")
        print(" ".join(map(str, cmd)))

        env = os.environ.copy()
        env["PYTHONPATH"] = str(TARGET_SITE) + (":" + env.get("PYTHONPATH", "") if env.get("PYTHONPATH") else "")
        env["HF_HUB_OFFLINE"] = "1"
        env["TRANSFORMERS_OFFLINE"] = "1"
        env["HF_DATASETS_OFFLINE"] = "1"
        env["TIKTOKEN_ENCODINGS_BASE"] = os.environ["TIKTOKEN_ENCODINGS_BASE"]
        env.pop("TIKTOKEN_RS_CACHE_DIR", None)

        self.proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
            start_new_session=True,
        )

        self._server_log_lines = []
        self._server_log_thread = threading.Thread(target=self._read_server_log, daemon=True)
        self._server_log_thread.start()

    def _read_server_log(self):
        try:
            for line in self.proc.stdout:
                self._server_log_lines.append(line.rstrip())
                if len(self._server_log_lines) > 5000:
                    self._server_log_lines = self._server_log_lines[-3000:]
                if (
                    "error" in line.lower()
                    or "traceback" in line.lower()
                    or "running on" in line.lower()
                    or "started server process" in line.lower()
                    or "application startup complete" in line.lower()
                    or "model" in line.lower()
                ):
                    print("[vLLM]", line.rstrip())
        except Exception as exc:
            print("[ОШИБКА ЧТЕНИЯ ЛОГА vLLM]", exc)

    def _wait_for_server(self):
        import urllib.request

        deadline = time.time() + 900
        url = f"http://127.0.0.1:{self.port}/v1/models"

        while time.time() < deadline:
            if self.proc.poll() is not None:
                print("\nХВОСТ ЛОГА СЕРВЕРА")
                print("\n".join(self._server_log_lines[-200:]))
                raise RuntimeError("vLLM server exited early")

            try:
                with urllib.request.urlopen(url, timeout=2) as r:
                    body = r.read().decode("utf-8", errors="replace")
                    print("сервер vLLM готов. /v1/models:")
                    print(body[:2000])
                    return
            except Exception:
                time.sleep(2)

        print("\nХВОСТ ЛОГА СЕРВЕРА")
        print("\n".join(self._server_log_lines[-200:]))
        raise TimeoutError("vLLM server did not become ready")

    def close(self):
        if self.proc is not None:
            with contextlib.suppress(Exception):
                os.killpg(os.getpgid(self.proc.pid), signal.SIGTERM)
            with contextlib.suppress(Exception):
                self.proc.wait(timeout=30)
            with contextlib.suppress(Exception):
                os.killpg(os.getpgid(self.proc.pid), signal.SIGKILL)
        self.proc = None

    def _build_messages(self, problem: str, exp_cfg: ExperimentConfig):
        system_content = SystemContent.new()
        system_content = system_content.with_model_identity(SYSTEM_IDENTITY)
        system_content = system_content.with_reasoning_effort(
            reasoning_effort=reasoning_effort_from_str(exp_cfg.reasoning_effort)
        )

        if bool(exp_cfg.use_tools):
            tool_config = ToolNamespaceConfig(
                name="python",
                description=exp_cfg.tool_prompt,
                tools=[],
            )
            system_content = system_content.with_tools(tool_config)

        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        developer_message = Message.from_role_and_content(Role.DEVELOPER, DEVELOPER_PROMPT)
        user_message = Message.from_role_and_content(Role.USER, problem)

        return [system_message, developer_message, user_message]

    def _render_prompt_ids(self, messages):
        convo = make_conversation(messages)
        return list(self.encoding.render_conversation_for_completion(convo, Role.ASSISTANT))

    def _parse_completion_tokens_safe(self, token_buffer):
        try:
            return self.encoding.parse_messages_from_completion_tokens(
                token_buffer,
                Role.ASSISTANT,
                strict=False,
            )
        except TypeError:
            return self.encoding.parse_messages_from_completion_tokens(
                token_buffer,
                Role.ASSISTANT,
            )

    def _make_tool_response(self, output: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name="python")
        msg = Message(author=author, content=[content]).with_recipient("assistant")
        if channel:
            msg = msg.with_channel(channel)
        return msg

    def _make_stop_kwargs(
        self,
        *,
        model_name: str,
        mode: str,
        prompt_ids,
        max_tokens: int,
        seed: int,
        temperature: float,
        top_p: float,
    ):
        extra_body = {
            "return_token_ids": True,
            "skip_special_tokens": False,
        }

        kwargs = dict(
            model=model_name,
            prompt=prompt_ids,
            max_tokens=int(max_tokens),
            temperature=float(temperature),
            top_p=float(top_p),
            seed=int(seed),
            stream=True,
            extra_body=extra_body,
        )

        if mode == "strings":
            kwargs["stop"] = self.stop_strings

        elif mode == "strings_include":
            kwargs["stop"] = self.stop_strings
            extra_body["include_stop_str_in_output"] = True

        elif mode == "token_ids":
            extra_body["stop_token_ids"] = self.stop_token_ids

        else:
            raise ValueError(f"Unknown stop mode: {mode}")

        return kwargs

    def _collect_stream(self, stream):
        token_buffer = []
        streamed_text = ""
        finish_reason = None

        try:
            for chunk in stream:
                if not getattr(chunk, "choices", None):
                    continue

                choice = chunk.choices[0]

                token_ids = getattr(choice, "token_ids", None) or []
                text = getattr(choice, "text", "") or ""
                fr = getattr(choice, "finish_reason", None)

                if token_ids:
                    token_buffer.extend(token_ids)
                if text:
                    streamed_text += text
                if fr is not None:
                    finish_reason = fr

        finally:
            with contextlib.suppress(Exception):
                stream.close()

        return token_buffer, streamed_text, finish_reason

    def _test_stop_mode_once(self, model_name: str, mode: str):
        rec = {
            "model_name": model_name,
            "mode": mode,
            "ok": False,
            "n_tokens": 0,
            "n_messages": 0,
            "last_channel": None,
            "last_recipient": None,
            "error": "",
            "text_tail": "",
        }

        try:
            exp_cfg = ExperimentConfig(
                label="stop_preflight",
                served_model_name=model_name,
                use_tools=False,
                reasoning_effort="medium",
                context_window=CONTEXT_WINDOW,
                generation_cap=GENERATION_CAP,
                samples=1,
            )

            messages = self._build_messages(
                "Compute 2+2. Put the final answer inside \\boxed{...}. Do not use tools.",
                exp_cfg,
            )
            prompt_ids = self._render_prompt_ids(messages)

            kwargs = self._make_stop_kwargs(
                model_name=model_name,
                mode=mode,
                prompt_ids=prompt_ids,
                max_tokens=STOP_PREFLIGHT_MAX_TOKENS,
                seed=123456,
                temperature=0.0,
                top_p=1.0,
            )

            stream = self.client.completions.create(**kwargs)
            token_buffer, text, finish_reason = self._collect_stream(stream)

            rec["n_tokens"] = len(token_buffer)
            rec["finish_reason"] = finish_reason
            rec["text_tail"] = text[-500:]

            if not token_buffer:
                rec["error"] = "empty token_buffer"
                return rec

            messages_out = self._parse_completion_tokens_safe(token_buffer)
            rec["n_messages"] = len(messages_out)

            if not messages_out:
                rec["error"] = "no parsed messages"
                return rec

            last = messages_out[-1]
            rec["last_channel"] = getattr(last, "channel", None)
            rec["last_recipient"] = getattr(last, "recipient", None)

            if rec["last_channel"] not in {"analysis", "commentary", "final"}:
                rec["error"] = f"unexpected channel: {rec['last_channel']}"
                return rec

            rec["ok"] = True
            return rec

        except Exception as exc:
            rec["error"] = f"{type(exc).__name__}: {exc}"
            return rec

    def assert_strings_stop_mode_works(self, model_name: str):
        rec = self._test_stop_mode_once(model_name, "strings")

        print(
            "[ПРОВЕРКА STOP-STRINGS]",
            "модель=", model_name,
            "ок=", rec["ok"],
            "токены=", rec["n_tokens"],
            "сообщения=", rec["n_messages"],
            "канал=", rec["last_channel"],
            "получатель=", rec["last_recipient"],
            "причина_завершения=", rec.get("finish_reason"),
            "ошибка=", rec["error"][:300],
        )

        assert rec["ok"], (
            "STOP_MODE='strings' failed Harmony parsing preflight for "
            f"{model_name}. Details: {rec}. "
            "If you accept fallback behavior, set STOP_MODE='auto_tinker'."
        )

        return rec

    def get_stop_mode_for_model(self, model_name: str):
        if STOP_MODE != "auto_tinker":
            return STOP_MODE

        if model_name in self.stop_mode_by_model:
            return self.stop_mode_by_model[model_name]

        results = []

        for mode in STOP_PREF_ORDER:
            rec = self._test_stop_mode_once(model_name, mode)
            results.append(rec)

            print(
                "[ПРЕДПРОВЕРКА STOP]",
                "модель=", model_name,
                "режим=", rec["mode"],
                "ок=", rec["ok"],
                "токены=", rec["n_tokens"],
                "сообщения=", rec["n_messages"],
                "канал=", rec["last_channel"],
                "получатель=", rec["last_recipient"],
                "ошибка=", rec["error"][:200],
            )

            if rec["ok"]:
                self.stop_mode_by_model[model_name] = mode
                print("[ПРЕДПРОВЕРКА STOP] выбран для", model_name, ":", mode)
                return mode

        raise RuntimeError(f"No STOP_MODE passed Harmony parsing preflight for {model_name}: {results}")

    def _stream_one_turn(self, exp_cfg: ExperimentConfig, prompt_ids, seed, budget_remaining):
        available = exp_cfg.context_window - len(prompt_ids) - CONTEXT_RESERVE
        if available < 64 or budget_remaining <= 0:
            return None, available

        max_tokens = min(int(budget_remaining), int(available))
        if max_tokens < 64:
            return None, available

        stop_mode = self.get_stop_mode_for_model(exp_cfg.served_model_name)

        kwargs = self._make_stop_kwargs(
            model_name=exp_cfg.served_model_name,
            mode=stop_mode,
            prompt_ids=prompt_ids,
            max_tokens=max_tokens,
            seed=seed,
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )

        stream = self.client.completions.create(**kwargs)
        return stream, available

    def run_sample(self, row: dict, exp_cfg: ExperimentConfig, sample_idx: int):
        problem_id = str(row["problem_id"])
        gold = int(row["answer"])
        seed = stable_seed(problem_id, sample_idx)

        start = time.time()

        sandbox = SubprocessPythonSandbox(timeout=PYTHON_TIMEOUT) if exp_cfg.use_tools else None

        messages = self._build_messages(row["problem"], exp_cfg)

        assistant_turn_texts = []
        assistant_tokens = 0
        turns = 0
        python_calls = 0
        python_errors = 0
        tool_events = []

        reason = None
        hard_truncated = False
        context_hit = False
        cap_hit = False
        timeout_hit = False
        parse_error = ""

        try:
            while True:
                if time.time() - start > ATTEMPT_TIMEOUT_SECONDS:
                    timeout_hit = True
                    hard_truncated = True
                    reason = "timeout"
                    break

                if turns >= exp_cfg.max_turns:
                    hard_truncated = True
                    reason = "max_turns"
                    break

                if assistant_tokens >= exp_cfg.generation_cap:
                    hard_truncated = True
                    cap_hit = True
                    reason = "generation_cap"
                    break

                prompt_ids = self._render_prompt_ids(messages)
                budget_remaining = exp_cfg.generation_cap - assistant_tokens

                stream, available = self._stream_one_turn(
                    exp_cfg,
                    prompt_ids,
                    seed=seed + turns,
                    budget_remaining=budget_remaining,
                )

                if stream is None:
                    hard_truncated = True
                    context_hit = available < 64
                    cap_hit = budget_remaining <= 0
                    reason = "context_window" if context_hit else "generation_cap"
                    break

                token_buffer, streamed_text, finish_reason = self._collect_stream(stream)

                turns += 1
                assistant_tokens += len(token_buffer)

                if not token_buffer:
                    hard_truncated = True
                    reason = "empty_generation"
                    break

                try:
                    new_messages = self._parse_completion_tokens_safe(token_buffer)
                except Exception as exc:
                    parse_error = f"{type(exc).__name__}: {exc}"
                    assistant_turn_texts.append(streamed_text)
                    hard_truncated = True
                    reason = "parse_error"
                    break

                if not new_messages:
                    parse_error = "no parsed messages"
                    assistant_turn_texts.append(streamed_text)
                    hard_truncated = True
                    reason = "parse_error"
                    break

                messages.extend(new_messages)

                for msg in new_messages:
                    txt = harmony_message_text(msg)
                    if txt:
                        assistant_turn_texts.append(txt)

                last = new_messages[-1]
                last_channel = getattr(last, "channel", None)
                last_recipient = getattr(last, "recipient", None)
                last_text = harmony_message_text(last)

                recipient_str = "" if last_recipient is None else str(last_recipient)
                is_python_call = recipient_str == "python"

                if last_channel == "final":
                    reason = "final"
                    break

                if exp_cfg.use_tools and is_python_call:
                    python_calls += 1
                    assert sandbox is not None

                    output, flags = sandbox.execute(last_text, timeout=PYTHON_TIMEOUT)
                    flags.update({
                        "call_idx": python_calls,
                        "output_head": output[:500],
                        "output_tail": output[-500:],
                        "output_len": len(output),
                    })
                    tool_events.append(flags)

                    if flags.get("is_error"):
                        python_errors += 1

                    messages.append(
                        self._make_tool_response(
                            output=output,
                            channel=last_channel,
                        )
                    )
                    continue

                if finish_reason == "length":
                    hard_truncated = True
                    reason = "truncated_generation"
                    break

                continue

            full_text = "\n".join(assistant_turn_texts)
            pred_strict = extract_answer_strict(full_text)
            pred_loose = extract_answer_loose(full_text)
            correct_strict = int(pred_strict is not None and int(pred_strict) == gold)
            correct_loose = int(pred_loose is not None and int(pred_loose) == gold)

            return {
                "config": exp_cfg.label,
                "served_model_name": exp_cfg.served_model_name,
                "stop_mode": self.get_stop_mode_for_model(exp_cfg.served_model_name),

                "problem_id": problem_id,
                "sample_idx": int(sample_idx),
                "gold_answer": gold,

                "pred_answer_strict": pred_strict,
                "is_correct_strict": correct_strict,
                "pred_answer_loose": pred_loose,
                "is_correct_loose": correct_loose,

                "pred_answer": pred_strict,
                "is_correct": correct_strict,

                "total_tokens": int(assistant_tokens),
                "turns": int(turns),
                "python_calls": int(python_calls),
                "python_errors": int(python_errors),
                "hard_truncated": int(hard_truncated),
                "context_hit": int(context_hit),
                "cap_hit": int(cap_hit),
                "timeout_hit": int(timeout_hit),
                "reason": reason,
                "parse_error": parse_error,
                "sample_valid": int(reason != "parse_error" and reason != "empty_generation"),

                "tool_events_count": int(len(tool_events)),
                "tool_error_count_audited": int(sum(bool(e.get("is_error", False)) for e in tool_events)),
                "tool_rewrite_last_expr_count": int(sum(bool(e.get("rewrote_last_expr", False)) for e in tool_events)),
                "tool_no_output_count": int(sum(bool(e.get("is_no_output", False)) for e in tool_events)),
                "tool_sentinel_leak_count": int(sum(bool(e.get("had_harmony_sentinel", False)) for e in tool_events)),
                "tool_fence_count": int(sum(bool(e.get("had_markdown_fence", False)) for e in tool_events)),

                "trajectory_text": full_text,
                "tool_events": tool_events,
                "run_error": "",
            }

        except Exception as exc:
            try:
                stop_mode = self.get_stop_mode_for_model(exp_cfg.served_model_name)
            except Exception:
                stop_mode = "unknown"

            return {
                "config": exp_cfg.label,
                "served_model_name": exp_cfg.served_model_name,
                "stop_mode": stop_mode,

                "problem_id": problem_id,
                "sample_idx": int(sample_idx),
                "gold_answer": gold,
                "pred_answer_strict": None,
                "is_correct_strict": 0,
                "pred_answer_loose": None,
                "is_correct_loose": 0,
                "pred_answer": None,
                "is_correct": 0,

                "total_tokens": 0,
                "turns": int(turns),
                "python_calls": int(python_calls),
                "python_errors": int(python_errors) + 1,
                "hard_truncated": 1,
                "context_hit": 0,
                "cap_hit": 0,
                "timeout_hit": 0,
                "reason": "run_error",
                "parse_error": "",
                "sample_valid": 0,

                "tool_events_count": int(len(tool_events)),
                "tool_error_count_audited": int(sum(bool(e.get("is_error", False)) for e in tool_events)),
                "tool_rewrite_last_expr_count": int(sum(bool(e.get("rewrote_last_expr", False)) for e in tool_events)),
                "tool_no_output_count": int(sum(bool(e.get("is_no_output", False)) for e in tool_events)),
                "tool_sentinel_leak_count": int(sum(bool(e.get("had_harmony_sentinel", False)) for e in tool_events)),
                "tool_fence_count": int(sum(bool(e.get("had_markdown_fence", False)) for e in tool_events)),

                "trajectory_text": "",
                "tool_events": tool_events,
                "run_error": f"{type(exc).__name__}: {exc}",
            }

        finally:
            if sandbox is not None:
                sandbox.close()

In [ ]:
def summarize_samples(samples_df: pd.DataFrame, eval_df: pd.DataFrame, exp_cfg: ExperimentConfig):
    problem_rows = []

    if "pred_answer_strict" not in samples_df.columns and "pred_answer" in samples_df.columns:
        samples_df = samples_df.copy()
        samples_df["pred_answer_strict"] = samples_df["pred_answer"]
    if "is_correct_strict" not in samples_df.columns and "is_correct" in samples_df.columns:
        samples_df = samples_df.copy()
        samples_df["is_correct_strict"] = samples_df["is_correct"]
    if "pred_answer_loose" not in samples_df.columns:
        samples_df = samples_df.copy()
        samples_df["pred_answer_loose"] = samples_df.get("pred_answer_strict", None)
    if "is_correct_loose" not in samples_df.columns:
        samples_df = samples_df.copy()
        samples_df["is_correct_loose"] = samples_df.get("is_correct_strict", 0)

    for _, row in eval_df.iterrows():
        pid = str(row["problem_id"])
        gold = int(row["answer"])

        sdf = samples_df[samples_df["problem_id"].astype(str) == pid].copy()
        if len(sdf) == 0:
            continue

        strict_maj, strict_votes = majority_answer(sdf["pred_answer_strict"].tolist())
        loose_maj, loose_votes = majority_answer(sdf["pred_answer_loose"].tolist())

        strict_majority_correct = int(strict_maj is not None and int(strict_maj) == gold)
        loose_majority_correct = int(loose_maj is not None and int(loose_maj) == gold)

        py_calls_sum = float(sdf["python_calls"].sum()) if "python_calls" in sdf else 0.0
        py_errs_sum = float(sdf["python_errors"].sum()) if "python_errors" in sdf else 0.0
        audited_errs_sum = float(sdf["tool_error_count_audited"].sum()) if "tool_error_count_audited" in sdf else 0.0

        problem_rows.append({
            "config": exp_cfg.label,
            "served_model_name": exp_cfg.served_model_name,
            "stop_mode": str(sdf["stop_mode"].iloc[0]) if "stop_mode" in sdf and len(sdf) else "",
            "problem_id": pid,
            "gold_answer": gold,
            "n_samples_done": int(len(sdf)),

            "strict_majority_answer": strict_maj,
            "strict_majority_votes": int(strict_votes),
            "strict_majority_correct": strict_majority_correct,
            "strict_pass_at_k": int(sdf["is_correct_strict"].max()),
            "strict_sample_accuracy": float(sdf["is_correct_strict"].mean()),
            "strict_num_valid_answers": int(sdf["pred_answer_strict"].notna().sum()),

            "loose_majority_answer": loose_maj,
            "loose_majority_votes": int(loose_votes),
            "loose_majority_correct": loose_majority_correct,
            "loose_pass_at_k": int(sdf["is_correct_loose"].max()),
            "loose_sample_accuracy": float(sdf["is_correct_loose"].mean()),
            "loose_num_valid_answers": int(sdf["pred_answer_loose"].notna().sum()),

            "majority_answer": strict_maj,
            "majority_votes": int(strict_votes),
            "problem_correct": strict_majority_correct,
            "pass_at_k": int(sdf["is_correct_strict"].max()),
            "num_valid_answers": int(sdf["pred_answer_strict"].notna().sum()),
            "sample_accuracy": float(sdf["is_correct_strict"].mean()),

            "sample_valid_rate": float(sdf["sample_valid"].mean()) if "sample_valid" in sdf else None,
            "mean_tokens": float(sdf["total_tokens"].mean()),
            "median_tokens": float(sdf["total_tokens"].median()),
            "p90_tokens": float(sdf["total_tokens"].quantile(0.90)),
            "mean_turns": float(sdf["turns"].mean()),
            "mean_python_calls": float(sdf["python_calls"].mean()),
            "mean_python_errors": float(sdf["python_errors"].mean()),
            "tool_error_rate_logged": float(py_errs_sum / max(1.0, py_calls_sum)),
            "tool_error_rate_audited": float(audited_errs_sum / max(1.0, py_calls_sum)),
            "mean_tool_events": float(sdf["tool_events_count"].mean()) if "tool_events_count" in sdf else 0.0,
            "mean_tool_rewrite_last_expr": float(sdf["tool_rewrite_last_expr_count"].mean()) if "tool_rewrite_last_expr_count" in sdf else 0.0,
            "mean_tool_no_output": float(sdf["tool_no_output_count"].mean()) if "tool_no_output_count" in sdf else 0.0,
            "mean_tool_sentinel_leaks": float(sdf["tool_sentinel_leak_count"].mean()) if "tool_sentinel_leak_count" in sdf else 0.0,
            "mean_tool_fences": float(sdf["tool_fence_count"].mean()) if "tool_fence_count" in sdf else 0.0,
            "timeouts": int(sdf["timeout_hit"].sum()),
            "context_hits": int(sdf["context_hit"].sum()),
            "cap_hits": int(sdf["cap_hit"].sum()),
            "parse_errors": int((sdf["reason"] == "parse_error").sum()),
            "run_errors": int((sdf["reason"] == "run_error").sum()),
            "trunc_frac": float(sdf["hard_truncated"].mean()),
            "context_hit_frac": float((sdf["reason"] == "context_window").mean()),
            "response_cap_hit_frac": float((sdf["reason"] == "generation_cap").mean()),
            "truncated_generation_frac": float((sdf["reason"] == "truncated_generation").mean()),
            "run_error_frac": float((sdf["reason"] == "run_error").mean()),
        })

    problems_df = pd.DataFrame(problem_rows)

    py_calls_sum = float(samples_df["python_calls"].sum()) if len(samples_df) and "python_calls" in samples_df else 0.0
    py_errs_sum = float(samples_df["python_errors"].sum()) if len(samples_df) and "python_errors" in samples_df else 0.0
    audited_errs_sum = float(samples_df["tool_error_count_audited"].sum()) if len(samples_df) and "tool_error_count_audited" in samples_df else 0.0

    summary = {
        "config": exp_cfg.label,
        "served_model_name": exp_cfg.served_model_name,
        "stop_mode": problems_df["stop_mode"].iloc[0] if len(problems_df) else "",
        "k": int(exp_cfg.samples),
        "n_problems": int(len(problems_df)),
        "n_samples": int(len(samples_df)),

        "strict_majority_at_k": float(problems_df["strict_majority_correct"].mean()) if len(problems_df) else None,
        "strict_pass_at_k": float(problems_df["strict_pass_at_k"].mean()) if len(problems_df) else None,
        "strict_sample_accuracy": float(samples_df["is_correct_strict"].mean()) if len(samples_df) else None,

        "loose_majority_at_k": float(problems_df["loose_majority_correct"].mean()) if len(problems_df) else None,
        "loose_pass_at_k": float(problems_df["loose_pass_at_k"].mean()) if len(problems_df) else None,
        "loose_sample_accuracy": float(samples_df["is_correct_loose"].mean()) if len(samples_df) else None,

        "problem_accuracy": float(problems_df["problem_correct"].mean()) if len(problems_df) else None,
        "pass_at_k": float(problems_df["pass_at_k"].mean()) if len(problems_df) else None,
        "sample_accuracy": float(samples_df["is_correct_strict"].mean()) if len(samples_df) else None,

        "sample_valid_rate": float(samples_df["sample_valid"].mean()) if len(samples_df) and "sample_valid" in samples_df else None,
        "mean_problem_tokens": float(problems_df["mean_tokens"].mean()) if len(problems_df) else None,
        "mean_sample_tokens": float(samples_df["total_tokens"].mean()) if len(samples_df) else None,
        "median_sample_tokens": float(samples_df["total_tokens"].median()) if len(samples_df) else None,
        "mean_python_calls": float(samples_df["python_calls"].mean()) if len(samples_df) else None,
        "mean_python_errors": float(samples_df["python_errors"].mean()) if len(samples_df) else None,
        "tool_error_rate_logged": float(py_errs_sum / max(1.0, py_calls_sum)) if len(samples_df) else None,
        "tool_error_rate_audited": float(audited_errs_sum / max(1.0, py_calls_sum)) if len(samples_df) else None,
        "mean_tool_events": float(samples_df["tool_events_count"].mean()) if len(samples_df) and "tool_events_count" in samples_df else None,
        "mean_tool_rewrite_last_expr": float(samples_df["tool_rewrite_last_expr_count"].mean()) if len(samples_df) and "tool_rewrite_last_expr_count" in samples_df else None,
        "mean_tool_no_output": float(samples_df["tool_no_output_count"].mean()) if len(samples_df) and "tool_no_output_count" in samples_df else None,
        "mean_tool_sentinel_leaks": float(samples_df["tool_sentinel_leak_count"].mean()) if len(samples_df) and "tool_sentinel_leak_count" in samples_df else None,
        "mean_tool_fences": float(samples_df["tool_fence_count"].mean()) if len(samples_df) and "tool_fence_count" in samples_df else None,
        "timeout_rate": float(samples_df["timeout_hit"].mean()) if len(samples_df) else None,
        "cap_hit_rate": float(samples_df["cap_hit"].mean()) if len(samples_df) else None,
        "context_hit_rate": float(samples_df["context_hit"].mean()) if len(samples_df) else None,
        "parse_error_rate": float((samples_df["reason"] == "parse_error").mean()) if len(samples_df) else None,
        "run_error_rate": float((samples_df["reason"] == "run_error").mean()) if len(samples_df) else None,
        "trunc_frac": float(samples_df["hard_truncated"].mean()) if len(samples_df) else None,
        "truncated_generation_rate": float((samples_df["reason"] == "truncated_generation").mean()) if len(samples_df) else None,
    }

    return problems_df, summary


def run_experiment(runner: GPTOSSRunner, exp_cfg: ExperimentConfig, eval_df: pd.DataFrame):
    exp_out = OUTPUT_DIR / exp_cfg.label
    exp_out.mkdir(parents=True, exist_ok=True)

    jsonl_path = exp_out / "samples.jsonl"
    samples_csv_path = exp_out / "samples.csv"
    problems_csv_path = exp_out / "problems.csv"
    summary_csv_path = exp_out / "summary.csv"

    done = set()
    if jsonl_path.exists():
        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                    done.add((str(rec["problem_id"]), int(rec["sample_idx"])))
                except Exception:
                    pass

    print("\n" + "#" * 120)
    print("ЗАПУСК:", exp_cfg.label)
    print("имя модели:", exp_cfg.served_model_name)
    print("уже готово:", len(done))
    print("#" * 120)

    selected_stop_mode = runner.get_stop_mode_for_model(exp_cfg.served_model_name)
    print("выбранный STOP_MODE:", selected_stop_mode)

    tasks = []
    rows = [r.to_dict() for _, r in eval_df.iterrows()]

    for row in rows:
        for sample_idx in range(exp_cfg.samples):
            key = (str(row["problem_id"]), int(sample_idx))
            if key in done:
                continue
            tasks.append((row, sample_idx))

    print("оставшиеся сэмплы:", len(tasks))

    def _run_one(args):
        row, sample_idx = args
        return runner.run_sample(row, exp_cfg, sample_idx)

    if tasks:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [ex.submit(_run_one, t) for t in tasks]

            for fut in tqdm(as_completed(futures), total=len(futures), desc=exp_cfg.label):
                rec = fut.result()

                with jsonl_path.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")

                print(
                    f"{rec['problem_id']} s{rec['sample_idx']:02d}:",
                    f"строго={rec.get('pred_answer_strict')}",
                    f"слабо={rec.get('pred_answer_loose')}",
                    f"золото={rec['gold_answer']}",
                    f"ок={rec.get('is_correct_strict', rec.get('is_correct'))}",
                    f"токенов={rec['total_tokens']}",
                    f"ходов={rec['turns']}",
                    f"py={rec['python_calls']}",
                    f"причина={rec['reason']}",
                    f"ошибка={rec['run_error'][:80]}",
                )

    sample_records = []
    if jsonl_path.exists():
        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    sample_records.append(json.loads(line))

    samples_df = pd.DataFrame(sample_records)
    samples_df = samples_df[samples_df["config"] == exp_cfg.label].copy()

    csv_df = samples_df.drop(columns=["trajectory_text", "tool_events"], errors="ignore")
    csv_df.to_csv(samples_csv_path, index=False)

    problems_df, summary = summarize_samples(samples_df, eval_df, exp_cfg)

    problems_df.to_csv(problems_csv_path, index=False)
    pd.DataFrame([summary]).to_csv(summary_csv_path, index=False)

    print("\САММАРИ:", exp_cfg.label)
    display(pd.DataFrame([summary]))
    display(problems_df)

    return samples_df, problems_df, summary

In [ ]:
runner = GPTOSSRunner()

if STOP_MODE == "strings" and ASSERT_STRINGS_STOP_PREFLIGHT:
    for exp_cfg in EXPERIMENTS:
        runner.assert_strings_stop_mode_works(exp_cfg.served_model_name)

all_summaries = []
all_problem_tables = []

try:
    for exp_cfg in EXPERIMENTS:
        samples_df, problems_df, summary = run_experiment(runner, exp_cfg, eval_df)
        all_summaries.append(summary)
        all_problem_tables.append(problems_df)

finally:
    runner.close()

summary_df = pd.DataFrame(all_summaries)
summary_path = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False)

problems_all_df = pd.concat(all_problem_tables, ignore_index=True) if all_problem_tables else pd.DataFrame()
problems_all_path = OUTPUT_DIR / "problems_all.csv"
problems_all_df.to_csv(problems_all_path, index=False)

print("\nИТОГОВОЕ САММАРИ")
display(summary_df)

In [ ]:
display(summary_df)

if len(problems_all_df):
    cols = [
        "config",
        "served_model_name",
        "stop_mode",
        "problem_id",
        "gold_answer",
        "strict_majority_answer",
        "strict_majority_votes",
        "strict_majority_correct",
        "strict_pass_at_k",
        "strict_sample_accuracy",
        "loose_majority_answer",
        "loose_majority_votes",
        "loose_majority_correct",
        "loose_pass_at_k",
        "loose_sample_accuracy",
        "mean_tokens",
        "median_tokens",
        "p90_tokens",
        "mean_python_calls",
        "mean_python_errors",
        "context_hits",
        "cap_hits",
        "parse_errors",
        "run_errors",
    ]
    display(problems_all_df[[c for c in cols if c in problems_all_df.columns]])

print("файлы вывода:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(p, p.stat().st_size)